# sukuma-toolkit GPU benchmarks (Google Colab)

Runs the two benchmark steps that need a GPU, then saves every report to Google Drive:

1. **Paper baseline**: the Sukuma Voices Whisper Large v3 fine-tune (`sartifyllc/sukuma-voices-asr`, 6.2 GB).
2. **Omnilingual CTC 1B** (too big for an 8 GB CPU machine) and **`omniASR_LLM_7B_v2`**, the model behind Meta's 2.0% CER claim for Sukuma.

Both run on the same clips and use the same three-level scoring as the CPU runs in `reports/`.

**Runtime:** Runtime → Change runtime type → **A100** (or L4). Whisper fits on a T4; the 7B model needs about 30 GB of GPU memory for comfort, so use an A100 for part 2.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sukuma-toolkit-reports'
!mkdir -p "$OUT"

In [ ]:
!git clone -q https://github.com/hesronkija/sukuma-toolkit.git /content/sukuma-toolkit
%cd /content/sukuma-toolkit
!pip install -q -e ".[whisper]"

## 1. Paper baseline (Whisper Large v3 fine-tune)

About 10 minutes for both test sets on an A100. The paper reports 25.19% WER on Sukuma Voices.

In [ ]:
!python scripts/whisper_baseline.py -n 362 --batch-size 16
!python scripts/whisper_baseline.py --dataset sukuma-speech-corpus -n 400 --batch-size 16
!cp reports/whisper_* "$OUT"/

## 2. Omnilingual CTC 1B and LLM 7B (Meta's reported model)

`omnilingual-asr` installs its own PyTorch (2.8), so **restart the session after this install cell**
(Runtime → Restart session), then run the cells below it. The first run downloads about 30 GB of weights.

In [ ]:
!pip install -q -e ".[omnilingual]"

In [ ]:
%cd /content/sukuma-toolkit
OUT = '/content/drive/MyDrive/sukuma-toolkit-reports'
for model in ["omniASR_CTC_1B_v2", "omniASR_LLM_7B_v2"]:
    !python scripts/omnilingual_smoke_test.py -n 362 --model {model} --batch-size 8
    !python scripts/omnilingual_smoke_test.py --dataset sukuma-speech-corpus -n 400 --model {model} --batch-size 8
!cp reports/omnilingual_*1B* reports/omnilingual_*7B* "$OUT"/


## 3. Bring the results home

Every `.md`, `.tsv` and `.json` report is now in `My Drive/sukuma-toolkit-reports`. Copy them into
`reports/` in your local repo and commit them with the CPU results.